# Nemotron SFT Fine-Tuning, Inference & Submission

1. Mount Drive, prepare SFT data from `tot_synthetic.jsonl`
2. LoRA fine-tune NVIDIA-Nemotron-3-Nano-30B-A3B (4-bit QLoRA via Unsloth)
3. Run inference on `test.csv` (greedy decoding, matching challenge eval)
4. Package `submission.zip` with adapter weights only

## 0. Install dependencies

In [1]:
!pip install uv
!uv pip install -q --upgrade pyarrow
!uv pip install -q mamba-ssm causal-conv1d
!pip install unsloth

In [2]:
import os, shutil, json, re, csv, zipfile
from pathlib import Path
from collections import Counter

# ── Challenge constants ────────────────────────────────────────────
CHALLENGE_MODEL_ID = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
MAX_SEQ_LEN    = 8192
MAX_NEW_TOKENS = 7680
NUMERIC_REL_TOL = 0.01

# ── Training constants ─────────────────────────────────────────────
TRAIN_MAX_SEQ_LEN  = MAX_SEQ_LEN   # full 8192 context
TRAIN_GRAD_ACCUM   = 8

# ── Colab detection ────────────────────────────────────────────────
IS_COLAB = False
try:
    from google.colab import drive, files, userdata
    IS_COLAB = True
except ImportError:
    drive = files = userdata = None

if IS_COLAB and drive is not None and not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

# ── Paths ──────────────────────────────────────────────────────────
DRIVE_REPO = Path("/content/drive/MyDrive/nemotron-challenge")
WORK_DIR   = DRIVE_REPO if DRIVE_REPO.exists() else Path.cwd()
DATA_DIR   = WORK_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

local_model_dir = WORK_DIR / "hf_models"
if local_model_dir.exists() and (local_model_dir / "config.json").exists():
    MODEL_PATH = str(local_model_dir)
    print(f"Using LOCAL model: {MODEL_PATH}")
else:
    MODEL_PATH = CHALLENGE_MODEL_ID
    if local_model_dir.exists():
        print(f"WARNING: {local_model_dir} exists but has no config.json — downloading from HF hub instead")
    print(f"Using HF hub model: {MODEL_PATH}")

HF_CACHE = Path("/content/hf_cache") if IS_COLAB else (WORK_DIR / ".hf_cache")
HF_CACHE.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(HF_CACHE))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

hf_token = os.environ.get("HF_TOKEN", "")
if IS_COLAB and not hf_token:
    try: hf_token = userdata.get("HF_TOKEN") or ""
    except: hf_token = ""
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    try:
        from huggingface_hub import login
        login(token=hf_token, add_to_git_credential=False)
        print("HF login OK")
    except: pass

TRAIN_CSV  = DATA_DIR / "train.csv"
TEST_CSV   = DATA_DIR / "test.csv"
INPUT_FILE = DATA_DIR / "tot_synthetic.jsonl"
SFT_FILE   = DATA_DIR / "finetune_sft.jsonl"

print(f"\nWorking dir : {WORK_DIR}")
print(f"Data dir    : {DATA_DIR}")
print(f"Model source: {MODEL_PATH}")
print(f"\nData files:")
for f in sorted(DATA_DIR.iterdir()):
    print(f"  {f.name}")

Using HF hub model: nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16

Working dir : /content/drive/MyDrive/nemotron-challenge
Data dir    : /content/drive/MyDrive/nemotron-challenge/data
Model source: nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16

Data files:
  finetune_rl.jsonl
  finetune_sft.jsonl
  run_log.txt
  test.csv
  tot_synthetic.jsonl
  train.csv


## 2. Prepare SFT data

In [3]:
TRAIN_CHAR_BUDGET = 24_000
CONCLUSION_TEMPLATE = "\n\n### Final Answer\n$\\boxed{{{answer}}}$"


def extract_boxed(text: str) -> str | None:
    matches = re.findall(r'\\boxed\{([^}]*)\}', text)
    return matches[-1].strip() if matches else None


def answers_match(predicted: str | None, correct: str | None) -> bool:
    if not predicted or not correct:
        return False
    if predicted == correct:
        return True
    try:
        p, c = float(predicted), float(correct)
    except Exception:
        return False
    if c == 0:
        return abs(p - c) <= 1e-12
    return abs(p - c) / abs(c) <= NUMERIC_REL_TOL


def replace_last_boxed(text: str, correct: str) -> tuple[str, bool]:
    replacement = f"$\\boxed{{{correct}}}$"
    matches = list(re.finditer(r'\$?\\boxed\{[^}]*\}\$?', text))
    if not matches:
        return text.rstrip() + CONCLUSION_TEMPLATE.format(answer=correct), True
    last = matches[-1]
    changed = last.group(0) != replacement
    return text[:last.start()] + replacement + text[last.end():], changed


def trim_and_cap(reasoning: str, gt: str) -> str:
    if len(reasoning) <= TRAIN_CHAR_BUDGET:
        return reasoning.rstrip() + CONCLUSION_TEMPLATE.format(answer=gt)
    chunk = reasoning[:TRAIN_CHAR_BUDGET]
    cut = chunk.rfind("\n\n")
    if cut == -1:
        cut = TRAIN_CHAR_BUDGET
    trimmed = reasoning[:cut].rstrip()
    trimmed = re.sub(r'\$?\\boxed\{[^}]*$', '', trimmed).rstrip()
    return trimmed + CONCLUSION_TEMPLATE.format(answer=gt)


if SFT_FILE.exists():
    print(f"Using existing: {SFT_FILE}")
    with open(SFT_FILE) as f:
        sft_records = [json.loads(l) for l in f if l.strip()]
else:
    assert INPUT_FILE.exists(), f"Need {SFT_FILE.name} or {INPUT_FILE.name} in {DATA_DIR}"
    with open(INPUT_FILE) as f:
        records = [json.loads(l) for l in f if l.strip()]
    print(f"Building SFT from {len(records)} ToT records...")

    sft_records = []
    correct_count = 0
    for r in records:
        gt = r["answer"].strip()
        reasoning = r["tot_reasoning"]
        predicted = extract_boxed(reasoning) or ""
        is_correct = answers_match(predicted, gt)
        if is_correct: correct_count += 1

        approx_tokens = len(reasoning) // 4
        if r.get("finish_reason") == "length" or approx_tokens > 7680:
            final = trim_and_cap(reasoning, gt)
        else:
            final, _ = replace_last_boxed(reasoning, gt)

        sft_records.append({
            "id": r["id"], "category": r["category"],
            "messages": [
                {"role": "user",      "content": r["messages"][0]["content"]},
                {"role": "assistant", "content": final},
            ],
            "answer": gt,
            "approx_tokens": len(final) // 4,
        })

    with open(SFT_FILE, "w") as f:
        for r in sft_records:
            f.write(json.dumps(r) + "\n")
    print(f"Original accuracy: {correct_count}/{len(records)} ({correct_count/len(records)*100:.1f}%)")

tok_lens = sorted(
    r.get("approx_tokens", len(r["messages"][1]["content"]) // 4)
    for r in sft_records
)
n = len(tok_lens)
print(f"SFT records : {n}")
print(f"Token lengths: min={tok_lens[0]}  median={tok_lens[n//2]}  max={tok_lens[-1]}")
print(f"Over budget  : {sum(1 for t in tok_lens if t > 7680)}")

Using existing: /content/drive/MyDrive/nemotron-challenge/data/finetune_sft.jsonl
SFT records : 391
Token lengths: min=123  median=376  max=7546
Over budget  : 0


## 3. Load model (4-bit QLoRA + Unsloth)

In [4]:
import torch
from unsloth import FastLanguageModel
from datasets import Dataset
from transformers import TrainingArguments, TrainerCallback
from trl import SFTTrainer


def assert_no_meta_tensors(model, *, stage: str):
    meta_tensors = []
    for name, tensor in model.named_parameters():
        if getattr(tensor, "is_meta", False):
            meta_tensors.append(name)
    for name, tensor in model.named_buffers():
        if getattr(tensor, "is_meta", False):
            meta_tensors.append(name)
    if meta_tensors:
        preview = ", ".join(meta_tensors[:5])
        raise RuntimeError(
            f"{stage}: found {len(meta_tensors)} meta tensors ({preview}). "
            "Model not fully materialized on GPU."
        )


def gib(b: int) -> float:
    return b / 1024**3


def print_gpu(label: str):
    if not torch.cuda.is_available(): return
    d = torch.cuda.current_device()
    print(
        f"[{label}] alloc={gib(torch.cuda.memory_allocated(d)):.2f} GiB  "
        f"reserved={gib(torch.cuda.memory_reserved(d)):.2f} GiB  "
        f"peak={gib(torch.cuda.max_memory_allocated(d)):.2f} GiB  "
        f"total={gib(torch.cuda.get_device_properties(d).total_memory):.2f} GiB"
    )


class GpuCallback(TrainerCallback):
    def on_train_begin(self, args, state, control, **kw):
        if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
        print_gpu("train_begin")
    def on_log(self, args, state, control, **kw):
        if state.global_step and state.global_step % args.logging_steps == 0:
            print_gpu(f"step_{state.global_step}")
    def on_train_end(self, args, state, control, **kw):
        print_gpu("train_end")


assert torch.cuda.is_available(), "CUDA not available — need GPU runtime"
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {gib(torch.cuda.get_device_properties(0).total_memory):.1f} GiB")

torch.cuda.empty_cache()

print(f"\nLoading: {MODEL_PATH}")
print(f"Config : 4-bit QLoRA, max_seq_len={TRAIN_MAX_SEQ_LEN}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name        = MODEL_PATH,
    max_seq_length    = TRAIN_MAX_SEQ_LEN,
    dtype             = None,
    load_in_4bit      = True,
    trust_remote_code = True,
)

assert_no_meta_tensors(model, stage="after model load")

# bitsandbytes 4-bit: model.device may report 'cpu' even when GPU is used.
param_devices = {str(p.device) for p in model.parameters()}
has_cuda = any("cuda" in d for d in param_devices)
print(f"Parameter devices: {param_devices}")
if not has_cuda:
    print("WARNING: No CUDA parameters found — attempting model.to('cuda')")
    model = model.to("cuda")

print(f"\nModel loaded — no meta tensors")
print_gpu("after_load")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU : NVIDIA A100-SXM4-80GB
VRAM: 79.3 GiB

Loading: nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16
Config : 4-bit QLoRA, max_seq_len=8192
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.3.18: Fast Nemotron_H patching. Transformers: 5.3.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Nemotron_H does not supp

modeling_nemotron_h.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16:
- modeling_nemotron_h.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/420 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16 does not have a padding token! Will use pad_token = <SPECIAL_999>.
Parameter devices: {'cuda:0'}

Model loaded — no meta tensors
[after_load] alloc=58.84 GiB  reserved=58.85 GiB  peak=58.84 GiB  total=79.25 GiB


## 4. Attach LoRA

In [5]:
LORA_RANK    = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.0     # must be 0 for Unsloth fast patching
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

assert LORA_RANK <= 32, f"Challenge requires rank <= 32, got {LORA_RANK}"

model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_RANK,
    target_modules             = TARGET_MODULES,
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = LORA_DROPOUT,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 42,
)

assert_no_meta_tensors(model, stage="after LoRA attach")

total = sum(p.numel() for p in model.parameters())
train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable: {train:,} / {total:,}  ({train/total*100:.2f}%)")
print_gpu("after_lora")

Unsloth: Detected MoE model with num_experts = 128 and target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']. Enabling LoRA on MoE parameters: ['mlp.experts.gate_up_proj', 'mlp.experts.down_proj']
Unsloth: PEFT set target_parameters but found no matching parameters.
This is expected for MoE models - Unsloth handles MoE expert LoRA targeting separately.
Unsloth: Making `model.base_model.model.backbone` require gradients
Trainable: 434,659,328 / 32,012,596,672  (1.36%)
[after_lora] alloc=60.46 GiB  reserved=60.47 GiB  peak=60.46 GiB  total=79.25 GiB


In [6]:
# Format: raw prompt continuation matching how vLLM feeds puzzles at eval time.
# {prompt}\n\n{response}{eos}

def to_raw_prompt(r):
    prompt   = r["messages"][0]["content"].rstrip()
    response = r["messages"][1]["content"].rstrip()
    eos = tokenizer.eos_token or ""
    return {"text": f"{prompt}\n\n{response}{eos}"}

dataset = Dataset.from_list(sft_records)
dataset = dataset.map(to_raw_prompt, remove_columns=dataset.column_names)

split = dataset.train_test_split(test_size=0.05, seed=42)
train_ds = split["train"]
eval_ds  = split["test"]

print(f"Train: {len(train_ds)}  |  Eval: {len(eval_ds)}")
print(f"Sample text (first 200 chars):\n{train_ds[0]['text'][:200]}...")

Map:   0%|          | 0/391 [00:00<?, ? examples/s]

Train: 371  |  Eval: 20
Sample text (first 200 chars):
[Hint: Compute the conversion factor from two examples, verify on a third, then apply to the target.]

In Alice's Wonderland, a secret unit conversion is applied to measurements. For example:
40.62 m ...


## 5. Fine-tune

In [7]:
# ── Monkey-patch fix_untrained_tokens for 4-bit compat ────────────
import sys
import unsloth_zoo.tokenizer_utils as _tku

_orig_fix = _tku.fix_untrained_tokens

@torch.inference_mode()
def _safe_fix_untrained(*args, **kwargs):
    try:
        return _orig_fix(*args, **kwargs)
    except (NotImplementedError, RuntimeError) as e:
        if "meta tensor" in str(e).lower() or "cannot copy" in str(e).lower():
            print(f"Skipping fix_untrained_tokens (4-bit compat): {type(e).__name__}")
        else:
            raise

_tku.fix_untrained_tokens = _safe_fix_untrained
for _mod in list(sys.modules.values()):
    try:
        if getattr(_mod, 'fix_untrained_tokens', None) is _orig_fix:
            _mod.fix_untrained_tokens = _safe_fix_untrained
    except Exception:
        pass

# ── Training ──────────────────────────────────────────────────────
# Model loaded at 8192 for inference. Training seq len capped to fit
# in VRAM — the Mamba2 activation memory scales steeply with seq len.
# Data: median=376, max=7546 tokens. 2048 covers >95% without truncation.
SFT_MAX_SEQ_LEN = 2048

# Free cached memory before trainer init
torch.cuda.empty_cache()
import gc; gc.collect()
print_gpu("before_trainer")

OUTPUT_DIR = Path("outputs/lora_adapter")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir                  = str(OUTPUT_DIR),
    num_train_epochs            = 3,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = TRAIN_GRAD_ACCUM,
    learning_rate               = 2e-4,
    lr_scheduler_type           = "cosine",
    warmup_ratio                = 0.05,
    optim                       = "paged_adamw_8bit",
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    logging_steps               = 10,
    eval_strategy               = "steps",
    eval_steps                  = 50,
    save_strategy               = "steps",
    save_steps                  = 100,
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    report_to                   = "none",
    seed                        = 42,
    dataloader_num_workers      = 0
)

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_ds,
    eval_dataset       = eval_ds,
    dataset_text_field = "text",
    max_seq_length     = SFT_MAX_SEQ_LEN,
    args               = training_args,
    callbacks          = [GpuCallback()],
)

print(f"Training: SFT max_seq_len={SFT_MAX_SEQ_LEN}, model context={TRAIN_MAX_SEQ_LEN}")
print("Starting training...")
trainer.train()
print("Training complete.")
print_gpu("after_training")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


[before_trainer] alloc=60.46 GiB  reserved=60.47 GiB  peak=60.46 GiB  total=79.25 GiB


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/371 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/20 [00:00<?, ? examples/s]

Training: SFT max_seq_len=2048, model context=8192
Starting training...


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 11}.


[train_begin] alloc=60.46 GiB  reserved=60.47 GiB  peak=60.46 GiB  total=79.25 GiB


Step,Training Loss,Validation Loss
50,2.204671,0.244785
100,1.912821,0.237665


[step_10] alloc=61.30 GiB  reserved=72.70 GiB  peak=71.62 GiB  total=79.25 GiB
[step_20] alloc=61.30 GiB  reserved=72.78 GiB  peak=71.62 GiB  total=79.25 GiB
[step_30] alloc=61.30 GiB  reserved=72.78 GiB  peak=71.62 GiB  total=79.25 GiB
[step_40] alloc=61.30 GiB  reserved=72.78 GiB  peak=71.62 GiB  total=79.25 GiB
[step_50] alloc=61.30 GiB  reserved=72.80 GiB  peak=71.62 GiB  total=79.25 GiB
[step_50] alloc=61.30 GiB  reserved=65.12 GiB  peak=71.62 GiB  total=79.25 GiB
[step_60] alloc=61.30 GiB  reserved=72.68 GiB  peak=71.62 GiB  total=79.25 GiB
[step_70] alloc=61.30 GiB  reserved=72.72 GiB  peak=71.62 GiB  total=79.25 GiB
[step_80] alloc=61.30 GiB  reserved=72.72 GiB  peak=71.62 GiB  total=79.25 GiB
[step_90] alloc=61.30 GiB  reserved=72.72 GiB  peak=71.62 GiB  total=79.25 GiB
[step_100] alloc=61.30 GiB  reserved=72.72 GiB  peak=71.62 GiB  total=79.25 GiB
[step_100] alloc=61.30 GiB  reserved=65.12 GiB  peak=71.62 GiB  total=79.25 GiB
[step_110] alloc=61.30 GiB  reserved=72.70 GiB  pe

## 7. Save adapter & create submission.zip

In [8]:
adapter_dir = OUTPUT_DIR / "final_adapter"
adapter_dir.mkdir(exist_ok=True)

# Save standard PEFT LoRA adapter
model.save_pretrained(str(adapter_dir))

print("Adapter files:")
for f in sorted(adapter_dir.iterdir()):
    print(f"  {f.name:<40} {f.stat().st_size/1024/1024:.1f} MB")

ALLOWED_FILES  = {"adapter_model.safetensors", "adapter_config.json"}
SUBMISSION_ZIP = Path("submission.zip")

with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in adapter_dir.iterdir():
        if f.name in ALLOWED_FILES:
            zf.write(f, arcname=f.name)
            print(f"  Zipped: {f.name}")

print(f"\nsubmission.zip: {SUBMISSION_ZIP.stat().st_size/1024/1024:.1f} MB")
with zipfile.ZipFile(SUBMISSION_ZIP, "r") as zf:
    for info in zf.infolist():
        print(f"  {info.filename:<40} {info.file_size/1024/1024:.1f} MB")

Adapter files:
  README.md                                0.0 MB
  adapter_config.json                      0.0 MB
  adapter_model.safetensors                1659.8 MB
  Zipped: adapter_model.safetensors
  Zipped: adapter_config.json

submission.zip: 1527.7 MB
  adapter_model.safetensors                1659.8 MB
  adapter_config.json                      0.0 MB


## 6. Inference on test.csv (greedy, matching challenge eval)

In [ ]:
BOXED_RE = re.compile(r'\\boxed\{([^}]*)\}')


def extract_last_boxed(text: str) -> str | None:
    matches = BOXED_RE.findall(text)
    return matches[-1].strip() if matches else None


def run_inference(model, tokenizer, prompt: str) -> tuple[str, str | None]:
    """Match challenge eval: raw prompt in, greedy decode, extract boxed."""
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN,
    ).to(model.device)
    input_len = inputs["input_ids"].shape[1]

    FastLanguageModel.for_inference(model)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = MAX_NEW_TOKENS,
            do_sample      = False,
            num_beams      = 1,
            top_p          = 1.0,
            use_cache      = True,
            eos_token_id   = tokenizer.eos_token_id,
            pad_token_id   = tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
    return generated, extract_last_boxed(generated)


with open(TEST_CSV) as f:
    test_rows = list(csv.DictReader(f))
gt = {}
with open(TRAIN_CSV) as f:
    for r in csv.DictReader(f):
        gt[r["id"]] = r["answer"]

print(f"Test puzzles: {len(test_rows)}\n")

results = []
for i, row in enumerate(test_rows):
    gt_answer = gt.get(row["id"], "??")
    print(f"{'='*60}")
    print(f"Puzzle {i+1}/{len(test_rows)}  |  ID: {row['id']}  |  GT: {gt_answer}")
    print(f"{'='*60}")

    generated, predicted = run_inference(model, tokenizer, row["prompt"])

    correct = answers_match(predicted, gt_answer)
    print(f"Predicted : {predicted}")
    print(f"Expected  : {gt_answer}")
    print(f"Status    : {'CORRECT' if correct else 'WRONG'}")
    print(f"Tokens    : ~{len(generated) // 4}")
    print(f"\n--- First 800 chars ---")
    print(generated[:800])
    print()

    results.append({
        "id": row["id"], "predicted": predicted or "",
        "gt_answer": gt_answer, "correct": correct,
        "generated": generated,
    })

n_correct = sum(1 for r in results if r["correct"])
print(f"\n{'='*60}")
print(f"SFT ACCURACY: {n_correct}/{len(results)} ({n_correct/len(results)*100:.1f}%)")
print(f"{'='*60}")

Test puzzles: 3

Puzzle 1/3  |  ID: 00066667  |  GT: 10010111


Both `max_new_tokens` (=7680) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
with open(DATA_DIR / "sft_test_results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"Results    : {DATA_DIR / 'sft_test_results.json'}")
print(f"Submission : {Path.cwd() / SUBMISSION_ZIP}")

if IS_COLAB and files:
    try: files.download(str(SUBMISSION_ZIP))
    except: pass

print("\nDONE — upload submission.zip to Kaggle")